# 01. NLR 데이터셋 구조 및 품질 진단

이 노트북은 NLR 원본 파일 전체를 메모리에 적재하지 않고 파일 catalog를 생성합니다. Parquet은 footer와 제한된 batch, CSV/log는 제한된 표본 행만 읽습니다. 결과는 이후 workload 특성추출과 gray-box/DFL 입력 설계의 데이터 사전으로 사용합니다.

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

# Jupyter를 저장소 루트 또는 notebooks 폴더에서 실행하는 경우를 모두 지원합니다.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from nlr_data.catalog import build_catalog, save_catalog, summarize_catalog

pd.set_option('display.max_colwidth', 120)
plt.style.use('seaborn-v0_8-whitegrid')

## 1. 경로 및 검사 범위 설정

`MAX_FILES=None`은 모든 대상 파일을 cataloging합니다. 처음 동작만 확인할 때는 20 또는 50으로 변경할 수 있습니다.

In [ ]:
DATASET_ROOT = PROJECT_ROOT.parent / 'NLR_dataset'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'data_audit'
SAMPLE_ROWS = 2_048
MAX_FILES = 50  # 빠른 시험 실행 시 20 또는 50으로 변경

print(f'Project root : {PROJECT_ROOT}')
print(f'Dataset root : {DATASET_ROOT}')
print(f'Output dir   : {OUTPUT_DIR}')
assert DATASET_ROOT.is_dir(), f'NLR 데이터셋을 찾을 수 없습니다: {DATASET_ROOT}'

## 2. 파일 catalog 생성

분석 대상은 `00_raw_datasets`, `01_aggregated_datasets`, `03_whole-facility_profiles`입니다. 분석 스크립트와 그림, `.git`, `.venv`는 제외합니다.

In [ ]:
catalog = build_catalog(
    DATASET_ROOT,
    sample_rows=SAMPLE_ROWS,
    max_files=MAX_FILES,
    show_progress=True,
)
saved_paths = save_catalog(catalog, OUTPUT_DIR)

print(f'Cataloged files: {len(catalog):,}')
for format_name, path in saved_paths.items():
    print(f'{format_name.upper():7s}: {path}')

catalog.head(10)

## 3. 데이터 그룹별 구성 및 검사 성공률

In [ ]:
summary = summarize_catalog(catalog)
summary

In [ ]:
quality_summary = (
    catalog.groupby(['dataset_group', 'inspection_status'], observed=True)
    .size()
    .rename('file_count')
    .reset_index()
)
quality_summary

오류가 있는 파일은 전체 catalog 생성을 중단시키지 않고 아래 표에 원인을 기록합니다.

In [ ]:
catalog.loc[
    catalog['inspection_status'].eq('error'),
    ['relative_path', 'extension', 'inspection_error'],
].head(30)

## 4. 파일 수와 데이터 용량 분포

In [ ]:
group_stats = (
    catalog.assign(file_size_mb=catalog['file_size_bytes'] / 1024**2)
    .groupby('dataset_group', observed=True)
    .agg(file_count=('relative_path', 'size'), total_size_mb=('file_size_mb', 'sum'))
    .sort_index()
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
group_stats['file_count'].plot.bar(ax=axes[0], color='#3B82F6', title='Files by dataset group')
group_stats['total_size_mb'].plot.bar(ax=axes[1], color='#10B981', title='Size by dataset group [MB]')
for axis in axes:
    axis.set_xlabel('')
    axis.tick_params(axis='x', rotation=25)
plt.tight_layout()
group_stats

## 5. Schema 변형 확인

동일한 workload 안에서 schema가 여러 개라면 특성추출 전에 reader 또는 column mapping을 분리해야 합니다.

In [ ]:
schema_summary = (
    catalog.loc[catalog['inspection_status'].eq('ok')]
    .groupby(
        ['dataset_group', 'workload_type', 'extension', 'schema_signature'],
        observed=True,
    )
    .agg(
        file_count=('relative_path', 'size'),
        example_file=('relative_path', 'first'),
        columns=('column_names', 'first'),
    )
    .reset_index()
    .sort_values(['dataset_group', 'workload_type', 'file_count'], ascending=[True, True, False])
)
schema_summary.head(30)

## 6. AIDC 모델링 활용 후보 변수

아래 표는 column 이름을 기준으로 한 1차 후보입니다. 실제 물리적 의미와 단위는 원 논문 및 생성 스크립트로 다시 검증해야 합니다.

In [ ]:
candidate_columns = catalog.loc[
    catalog['inspection_status'].eq('ok'),
    [
        'dataset_group',
        'workload_type',
        'source_kind',
        'relative_path',
        'primary_time_column',
        'sample_interval_seconds',
        'power_columns',
        'workload_columns',
        'temperature_columns',
    ],
].copy()

candidate_columns.head(30)

## 7. 다음 단계 판단 기준

Catalog 결과를 이용해 다음을 결정합니다.

1. workload 유형별로 사용할 대표 schema와 시간 해상도
2. 직접 측정 전력(raw/aggregated)과 DIPLOEE 기반 whole-facility 모의 전력의 분리
3. workload 입력, IT 전력 보정 target, 불확실성 산정 변수의 구분
4. ramp rate·burst·주기성 계산 전 필요한 resampling 및 단위 통일 규칙

이 판단이 끝난 후 `feature_extraction.py`를 구현합니다.